In [ ]:
import psutil
import time
import csv
from datetime import datetime
import os

# -------- SETTINGS --------
OUTPUT_FILE = "../data/system_metrics.csv"
INTERVAL = 1  # seconds

# -------- CREATE FILE WITH HEADER IF NOT EXISTS --------
file_exists = os.path.isfile(OUTPUT_FILE)

with open(OUTPUT_FILE, mode="a", newline="") as file:
    writer = csv.writer(file)

    if not file_exists:
        writer.writerow([
            "timestamp",
            "cpu_percent",
            "ram_percent",
            "net_bytes_per_sec"
        ])

    # -------- INITIAL NETWORK COUNTERS --------
    prev_net = psutil.net_io_counters()
    prev_time = time.time()

    print("Collecting data... Press Ctrl+C to stop.")

    try:
        while True:
            # Current time
            now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            # CPU and RAM
            cpu = psutil.cpu_percent(interval=None)
            ram = psutil.virtual_memory().percent

            # Network counters
            current_net = psutil.net_io_counters()
            current_time = time.time()

            # Calculate rate (bytes/sec)
            bytes_sent = current_net.bytes_sent - prev_net.bytes_sent
            bytes_recv = current_net.bytes_recv - prev_net.bytes_recv
            net_rate = (bytes_sent + bytes_recv) / (current_time - prev_time)

            # Write row
            writer.writerow([now, cpu, ram, net_rate])
            file.flush()  # ensure data is saved immediately

            # Update previous values
            prev_net = current_net
            prev_time = current_time

            # Wait for next sample
            time.sleep(INTERVAL)

    except KeyboardInterrupt:
        print("\nData collection stopped.")



Data collection stopped.
